# Redes de Petri Coloridas e Linguagens de Programação

As **Redes de Petri Coloridas (RPC/CPN)** descrevem o comportamento de sistemas através da dinâmica entre **lugares**, **tokens** e **transições** — um paradigma fundamentalmente diferente da programação procedural ou orientada a objetos. O sistema não executa uma sequência fixa de comandos: ele *evolui* conforme transições vão sendo habilitadas pelo estado atual da rede.

Uma confusão frequente entre iniciantes é associar o CPN IDE ao HTML, porque os modelos são salvos em XML/PNML. Esse formato é apenas o mecanismo de serialização da estrutura gráfica da rede — a lógica do modelo é escrita em **Standard ML (SML)**, uma linguagem funcional estrita com tipagem estática e inferência de tipos. SML pertence à mesma família de OCaml, F# e Haskell, e não guarda relação com HTML.

## Color Sets: tipagem dos tokens

Em SML/CPN, cada token carrega um valor de um tipo bem definido, chamado **color set**. Isso permite representar estados, leituras de sensores e comandos de forma semanticamente precisa — conceito análogo a `enum` e tipos algébricos em linguagens como Rust, OCaml e Ada.

In [ ]:
# Equivalente conceitual em Python — apenas para ilustração didática
# A sintaxe real é SML, executada no CPN IDE

# colset STATE = with Idle | Walk | Turn;   <- SML
# colset SENSOR = int;                      <- SML

from enum import Enum

class State(Enum):
    Idle = 0
    Walk = 1
    Turn = 2

# SENSOR é representado como int
sensor_value: int = 0

print("States:", [s.name for s in State])

## Guardas e funções de arco

As transições nas RPC podem ter **guardas** — condições booleanas que determinam se a transição está habilitada. Em SML, essas funções seguem um estilo declarativo: definem *o que* é verdadeiro, não *como* calcular passo a passo.

A comparação abaixo mostra a mesma lógica de detecção de obstáculo nas três linguagens. Note que em SML o corpo da função *é* a expressão — não há `return` explícito.

In [2]:
# Python
def detect_obstacle(x: int) -> bool:
    return x < 10

# Equivalente em SML (CPN IDE):
# fun detectObstacle(x:int) = x < 10;

# Equivalente em C:
# bool detectObstacle(int x) { return x < 10; }

# Teste
for dist in [5, 10, 15]:
    print(f"distância={dist}cm → obstáculo: {detect_obstacle(dist)}")

distância=5cm → obstáculo: True
distância=10cm → obstáculo: False
distância=15cm → obstáculo: False


A sintaxe é superficialmente próxima, mas o papel dessas funções nas RPC é diferente: elas não são chamadas explicitamente pelo programador, mas avaliadas *pelo motor de simulação* a cada tentativa de disparo de transição.

## Concorrência

Uma das vantagens centrais das RPC é a modelagem natural de **comportamento concorrente**. No [robô Otto](https://www.ottodiy.com/), por exemplo, a caminhada e a leitura do sensor ultrassônico ocorrem simultaneamente — situação difícil de representar com clareza em código procedural, mas direta em uma rede de Petri: dois fluxos de tokens coexistem na rede e interagem apenas quando uma transição compartilhada é habilitada.

Quando o sensor registra distância abaixo do limiar, um token com valor crítico alcança o lugar de entrada de uma transição `Walk → Turn`, que se habilita e redireciona o estado do robô. Não há `if` global nem loop de controle central — a sincronização emerge da topologia da própria rede.

In [3]:
# Simulação simplificada da dinâmica de tokens — apenas ilustração
# A rede real seria definida graficamente no CPN IDE

import random

state = "Walk"
steps = 0

print(f"{'Passo':<8} {'Sensor (cm)':<14} {'Estado'}")
print("-" * 35)

while steps < 8:
    sensor = random.randint(5, 25)
    if state == "Walk" and detect_obstacle(sensor):
        state = "Turn"      # transição Walk→Turn habilitada
    elif state == "Turn":
        state = "Walk"      # transição Turn→Walk habilitada após girar
    print(f"{steps:<8} {sensor:<14} {state}")
    steps += 1

Passo    Sensor (cm)    Estado
-----------------------------------
0        14             Walk
1        13             Walk
2        15             Walk
3        22             Walk
4        23             Walk
5        8              Turn
6        9              Walk
7        8              Turn


## Arquitetura: RPC + ESP32

Uma prática consolidada em robótica embarcada é usar as RPC como **camada de especificação formal** e o microcontrolador como **executor físico**. A rede descreve e valida o comportamento — estados, transições, condições de disparo — antes mesmo de qualquer linha de firmware ser escrita. O ESP32 recebe então uma especificação verificada formalmente e implementa os comandos concretos: controle dos servos, leitura do ultrassônico e comunicação.

Essa separação reduz erros de lógica concorrente e facilita a verificação de propriedades como ausência de deadlock e alcançabilidade de estados — análises que o CPN IDE realiza automaticamente sobre o modelo.